In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive_output, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, Output
from IPython.display import display

# ------------------------------------------------------------
# 1. DESCRIPTION
# ------------------------------------------------------------

description = HTML("""
<div style="
    border:1px solid #b9d7f5;
    border-radius:8px;
    padding:8px 10px;
    margin-bottom:10px;
    font-size:13px;
    line-height:1.35;
    background-color:#f7fbff;
">
<div><b>Purpose:</b> Compare the characteristic frequencies of second-order low-pass and high-pass analog filters.</div>
<div><b>What we see:</b> The normalized gain G(ω) as a function of angular frequency ω. The vertical lines indicate the natural frequency ω₀, the resonant-maximum frequency ωmax, when it exists, and the cutoff frequency ωc, where G(ω) = 1/√2.</div>
<div><b>What happens as we interact:</b> By changing Q and ω₀, we observe how the resonant peak, the cutoff frequency ωc, and their positions relative to ω₀ change. For Q &lt; 1/√2, no resonant maximum exists.</div>
</div>
""")

# ------------------------------------------------------------
# 2. CONTROLS
# ------------------------------------------------------------

filter_radio = RadioButtons(options=['Low-pass', 'High-pass'], value='Low-pass', description='', layout=Layout(width='160px'))

q_slider = FloatSlider(min=0.30, max=5.00, step=0.05, value=1.00, description='Q:', continuous_update=True, readout=True, readout_format='.2f', style={'description_width':'30px'}, layout=Layout(width='220px'))

omega0_slider = FloatSlider(min=1.00, max=5.00, step=0.10, value=3.00, description='ω₀:', continuous_update=True, readout=True, readout_format='.2f', style={'description_width':'30px'}, layout=Layout(width='220px'))

# ------------------------------------------------------------
# 3. CUSTOM LEGEND
# ------------------------------------------------------------

legend_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    width:125px;
    font-size:13px;
    line-height:1.7;
    background:white;
">

<div>
<span style="display:inline-block; width:32px; border-top:3px solid red; vertical-align:middle; margin-right:7px;"></span>
G(ω)
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dashed #888888; vertical-align:middle; margin-right:7px;"></span>
1/√2
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dotted black; vertical-align:middle; margin-right:7px;"></span>
ω₀
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dash-dot #888888; vertical-align:middle; margin-right:7px;"></span>
ωc
</div>

<div>
<span style="display:inline-block; width:32px; border-top:2px dashed black; vertical-align:middle; margin-right:7px;"></span>
ωmax
</div>

</div>
""")

filter_label = HTML("<div style='font-size:14px; font-weight:bold; margin-top:12px; margin-bottom:2px;'>Filter:</div>")

# ------------------------------------------------------------
# 4. OUTPUT WIDGETS
# ------------------------------------------------------------

plot_output = Output(layout=Layout(width='calc(100% - 240px)', overflow='visible'))
info_output = Output(layout=Layout(width='auto', overflow='visible'))

# ------------------------------------------------------------
# 5. MAIN PLOT FUNCTION
# ------------------------------------------------------------

def plot_characteristic_frequencies(filter_type, Q, omega0):

    omega = np.linspace(0.01, 10.0, 3000)

    r = omega / omega0

    denominator = np.sqrt((1.0 - r**2)**2 + (r / Q)**2)

    if filter_type == 'Low-pass':

        G = 1.0 / denominator

        omega_c = omega0 * np.sqrt((1.0 - 1.0 / (2.0 * Q**2)) + np.sqrt((1.0 - 1.0 / (2.0 * Q**2))**2 + 1.0))

        if Q >= 1.0 / np.sqrt(2.0):
            omega_max = omega0 * np.sqrt(1.0 - 1.0 / (2.0 * Q**2))
            G_max = Q / np.sqrt(1.0 - 1.0 / (4.0 * Q**2))
        else:
            omega_max = None
            G_max = None

        reference_gain = 'G(0) = 1'

        title = 'Second-Order Low-Pass Filter'

    else:

        G = r**2 / denominator

        omega_c = omega0 * np.sqrt(1.0 / (2.0 * Q**2) - 1.0 + np.sqrt((1.0 - 1.0 / (2.0 * Q**2))**2 + 1.0))

        if Q >= 1.0 / np.sqrt(2.0):
            omega_max = omega0 / np.sqrt(1.0 - 1.0 / (2.0 * Q**2))
            G_max = Q / np.sqrt(1.0 - 1.0 / (4.0 * Q**2))
        else:
            omega_max = None
            G_max = None

        reference_gain = 'G(∞) = 1'

        title = 'Second-Order High-Pass Filter'

    # --------------------------------------------------------
    # 6. PLOT
    # --------------------------------------------------------

    with plot_output:

        plot_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 4.6))

        ax.plot(omega, G, 'r-', linewidth=2.0)

        ax.axhline(1.0 / np.sqrt(2.0), color='gray', linestyle='--', linewidth=1.2)

        ax.axvline(omega0, color='black', linestyle=':', linewidth=1.5)

        ax.axvline(omega_c, color='gray', linestyle='-.', linewidth=1.5)

        if omega_max is not None and omega_max <= omega[-1]:
            ax.axvline(omega_max, color='black', linestyle='--', linewidth=1.5)
            ax.plot(omega_max, G_max, 'ko', markersize=4)

        ax.set_xlim(0.0, 10.0)

        ax.set_ylim(0.0, 6.0)

        ax.set_xlabel('Angular Frequency ω  (rad/s)', fontsize=11)

        ax.set_ylabel('Normalized Gain G(ω)', fontsize=11)

        ax.set_title(title, fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, linestyle=':', alpha=0.35)

        plt.tight_layout()

        plt.show()

        plt.close(fig)

    # --------------------------------------------------------
    # 7. INFORMATION FRAME
    # --------------------------------------------------------

    if omega_max is None:

        omega_max_text = '<span style="color:#555555;">none</span>'

    else:

        omega_max_text = f'<span style="color:#0066cc;">{omega_max:.2f} rad/s</span>'

    info_html = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:7px 11px;
        font-size:13px;
        display:flex;
        flex-direction:row;
        align-items:center;
        justify-content:flex-start;
        gap:24px;
        white-space:nowrap;
        background:white;
        width:fit-content;
    ">

        <div>
            <b>Filter:</b>
            <span style="color:#0066cc;">{filter_type}</span>
        </div>

        <div>
            <b>Q:</b>
            <span style="color:#0066cc;">{Q:.2f}</span>
        </div>

        <div>
            <b>ω₀:</b>
            <span style="color:#0066cc;">{omega0:.2f} rad/s</span>
        </div>

        <div>
            <b>ωmax:</b>
            {omega_max_text}
        </div>

        <div>
            <b>ωc:</b>
            <span style="color:#0066cc;">{omega_c:.2f} rad/s</span>
        </div>

        <div>
            <b>Reference gain:</b>
            <span style="color:#0066cc;">{reference_gain}</span>
        </div>

    </div>
    """

    with info_output:

        info_output.clear_output(wait=True)

        display(HTML(info_html))

# ------------------------------------------------------------
# 8. INTERACTION
# ------------------------------------------------------------

interactive_controls = interactive_output(plot_characteristic_frequencies, {'filter_type': filter_radio, 'Q': q_slider, 'omega0': omega0_slider})

interactive_controls.layout.display = 'none'

# ------------------------------------------------------------
# 9. LEFT CONTROL COLUMN
# ------------------------------------------------------------

control_column = VBox([legend_html, filter_label, filter_radio, q_slider, omega0_slider], layout=Layout(width='230px', min_width='230px', align_items='flex-start', padding='0px 0px 0px 4px'))

# ------------------------------------------------------------
# 10. MAIN AREA
# ------------------------------------------------------------

main_area = HBox([control_column, plot_output], layout=Layout(width='100%', align_items='flex-start', justify_content='flex-start', overflow='visible'))

# ------------------------------------------------------------
# 11. INFORMATION AREA
# ------------------------------------------------------------

info_spacer = HTML("", layout=Layout(width='230px', min_width='230px'))

info_area = HBox([info_spacer, info_output], layout=Layout(width='100%', align_items='flex-start', justify_content='flex-start', overflow='visible'))

# ------------------------------------------------------------
# 12. FINAL DISPLAY
# ------------------------------------------------------------

display(description)

display(main_area)

display(info_area)

display(interactive_controls)